In [ ]:
import os
import pandas as pd
from datasets import load_dataset
import json
import random

# 1. CRÉATION DE L'ARBORESCENCE (Structure TRADITECH 225)
folders = ["data/raw", "data/processed", "data/splits"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# 2. TÉLÉCHARGEMENT DES DONNÉES MASAKHANE
print("--- Étape 1 : Téléchargement de Masakhane (fra-bam) ---")
remote_dataset = load_dataset("masakhane/mafand", "fra-bam")
# On convertit en liste simple pour la manipulation
masakhane_list = [{"fra": x["fra"], "bam": x["bam"]} for x in remote_dataset["train"]["translation"]]

# 3. CHARGEMENT DE TES 522 MOTS LOCAUX
# Assure-toi d'avoir uploadé 'mon_lexique.csv' dans le dossier racine de Colab
print("--- Étape 2 : Chargement du lexique local ---")
try:
    local_df = pd.read_csv('mon_lexique.csv')
    local_list = [{"fra": row['francais'], "bam": row['dioula']} for _, row in local_df.iterrows()]
except FileNotFoundError:
    print("Attention : 'mon_lexique.csv' non trouvé. Seul Masakhane sera utilisé.")
    local_list = []

# 4. FUSION ET TRANSFORMATION EN FORMAT "CHATBOT"
print("--- Étape 3 : Fusion et Transformation ---")
all_data = masakhane_list + local_list
chatbot_data = []

for item in all_data:
    # Variante 1 : Question directe
    chatbot_data.append({
        "instruction": f"Comment dit-on en Dioula : {item['fra']} ?",
        "response": f"En Dioula, on dit : {item['bam']}."
    })
    # Variante 2 : Traduction simple
    chatbot_data.append({
        "instruction": f"Traduis en Dioula : {item['fra']}",
        "response": item['bam']
    })

# 5. SAUVEGARDE FINALE
random.shuffle(chatbot_data) # Mélange pour un meilleur apprentissage
with open("data/processed/final_chatbot_data.jsonl", "w", encoding="utf-8") as f:
    for entry in chatbot_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write("\n")

print(f"Terminé ! Ton dataset final contient {len(chatbot_data)} exemples dans data/processed/")